# 📊 Session 4: Trend Following & Crisis Alpha
# PRESENTATION GRAPHICS

## Commodities Club - Northeastern University

---

This notebook generates all visualizations for the PowerPoint presentation.

---

In [ ]:
!pip install yfinance pandas numpy matplotlib seaborn scipy -q

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns
import yfinance as yf
from datetime import date
from tqdm import tqdm
from scipy import stats

# Style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [14, 8]
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12

# Colors
COLORS = {
    'trend': '#1f77b4',      # Blue
    'spy': '#ff7f0e',        # Orange
    'blend': '#2ca02c',      # Green
    'blend2': '#9467bd',     # Purple
    'crisis': '#d62728',     # Red
    'positive': '#2ca02c',   # Green
    'negative': '#d62728',   # Red
}

print("✅ Ready for visualization")

In [ ]:
# Constants
DAYS = 252
ANN = np.sqrt(DAYS)
LOOKBACKS = [64, 128, 256]
REBAL_FREQ = 21
TREND_VOL = 0.10

CRISES = {
    'GFC\n2008-09': (date(2008, 9, 15), date(2009, 3, 9)),
    'Euro\n2011-12': (date(2011, 7, 1), date(2012, 1, 31)),
    'Oil Crash\n2014-15': (date(2014, 6, 20), date(2015, 3, 17)),
    'China\n2015-16': (date(2015, 8, 11), date(2016, 2, 11)),
    'COVID\n2020': (date(2020, 2, 19), date(2020, 3, 23)),
    'Rate Hikes\n2022': (date(2022, 1, 3), date(2022, 10, 12)),
}

UNIVERSE = [
    'QQQ', 'IWM', 'EFA', 'EEM', 'VGK', 'EWJ',
    'TLT', 'IEF', 'LQD', 'HYG',
    'GLD', 'SLV',
    'VNQ',
    'UUP', 'FXE',
]

print("✅ Config set")

In [ ]:
# Helper functions
def clean_tz(df):
    if hasattr(df.index, 'tz') and df.index.tz is not None:
        df.index = df.index.tz_localize(None)
    return df

def load_data(symbols, start, end):
    prices = {}
    for sym in tqdm(symbols, desc="Loading"):
        try:
            df = yf.Ticker(sym).history(start=start, end=end, auto_adjust=True)
            if len(df) >= 252:
                prices[sym] = clean_tz(df)['Close']
        except:
            pass
    return pd.DataFrame(prices).dropna(how='all').ffill(limit=5)

def ewma_vol(returns, span=36):
    vol = returns.ewm(span=span, min_periods=18).std() * ANN
    return vol.clip(lower=0.05, upper=0.80)

def tsmom_signal(prices, lookback):
    ret = prices.pct_change()
    log_ret = np.log1p(ret)
    cum = log_ret.rolling(lookback, min_periods=lookback//2).sum()
    vol = ewma_vol(ret)
    scale = np.sqrt(lookback / DAYS)
    return cum / (vol * scale + 1e-8)

def generate_forecasts(prices, lookbacks):
    forecasts = []
    for lb in lookbacks:
        raw = tsmom_signal(prices, lb)
        mean_abs = raw.abs().expanding(min_periods=252).mean()
        scalar = 10.0 / mean_abs.clip(lower=0.1)
        scalar = scalar.clip(upper=50)
        scaled = (raw * scalar).clip(-20, 20)
        forecasts.append(scaled)
    combined = sum(forecasts) / len(forecasts)
    fdm = 1.0 + 0.02 * len(lookbacks)
    return (combined * fdm).clip(-20, 20)

def size_positions(forecasts, prices, target_vol=TREND_VOL):
    ret = prices.pct_change()
    vol = ewma_vol(ret)
    forecasts, vol = forecasts.align(vol, join='inner')
    n = len(forecasts.columns)
    idm = min(1.0 + 0.03 * n, 2.0)
    weights = (target_vol / vol) * (forecasts / 10.0) * idm / n
    weights = weights.clip(-0.5, 0.5)
    gross = weights.abs().sum(axis=1)
    scale = (3.0 / gross).clip(upper=1.0)
    return weights.multiply(scale, axis=0)

def monthly_rebalance(weights, freq=REBAL_FREQ):
    monthly = weights.copy()
    last_weights = None
    for i in range(len(monthly)):
        if i % freq == 0 or last_weights is None:
            last_weights = monthly.iloc[i].copy()
        else:
            monthly.iloc[i] = last_weights
    return monthly

def backtest(prices, weights, tc_bps=10):
    common = prices.columns.intersection(weights.columns)
    prices = prices[common]
    weights = weights[common]
    ret = prices.pct_change()
    idx = ret.index.intersection(weights.index)
    ret = ret.loc[idx]
    weights = weights.loc[idx]
    lagged = weights.shift(1).fillna(0)
    gross = (lagged * ret).sum(axis=1)
    turn = lagged.diff().abs().sum(axis=1) / 2
    costs = turn * (tc_bps / 10000)
    return gross - costs, lagged

print("✅ Functions ready")

In [ ]:
# Load data and run strategy
print("Loading data...")
prices = load_data(UNIVERSE, '2007-01-01', '2024-12-31')
spy_df = load_data(['SPY'], '2007-01-01', '2024-12-31')
spy_ret = spy_df['SPY'].pct_change().dropna()

print("\nGenerating signals...")
forecasts = generate_forecasts(prices, LOOKBACKS)

print("Sizing positions...")
weights = size_positions(forecasts, prices)
weights = monthly_rebalance(weights)

print("Running backtest...")
trend_ret, positions = backtest(prices, weights)

# Align
df = pd.DataFrame({'trend': trend_ret, 'spy': spy_ret}).dropna()

# Create strategies
blend_70_30 = 0.7 * df['spy'] + 0.3 * df['trend']
blend_50_50 = 0.5 * df['spy'] + 0.5 * df['trend']

print("\n✅ Data ready for visualization")

---
# 📈 SLIDE 1: What is Time-Series Momentum (TSMOM)?
---

In [ ]:
# SLIDE 1: Signal Generation Concept
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Pick one asset to illustrate
asset = 'GLD'
asset_price = prices[asset].dropna()
asset_ret = asset_price.pct_change()

# 1. Price with trend
ax1 = axes[0, 0]
ax1.plot(asset_price.index, asset_price, 'k-', lw=1.5, label=f'{asset} Price')
ma_256 = asset_price.rolling(256).mean()
ax1.plot(ma_256.index, ma_256, 'r--', lw=2, label='256-day MA')
ax1.fill_between(asset_price.index, asset_price, ma_256, 
                  where=asset_price > ma_256, alpha=0.3, color='green', label='Long Signal')
ax1.fill_between(asset_price.index, asset_price, ma_256, 
                  where=asset_price < ma_256, alpha=0.3, color='red', label='Short Signal')
ax1.set_title('1. Price vs Moving Average → Trend Direction', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price ($)')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# 2. Raw momentum signal
ax2 = axes[0, 1]
mom_256 = np.log1p(asset_ret).rolling(256).sum()
ax2.plot(mom_256.index, mom_256, 'b-', lw=1.5)
ax2.axhline(0, color='black', lw=1)
ax2.fill_between(mom_256.index, mom_256, 0, where=mom_256 > 0, alpha=0.4, color='green')
ax2.fill_between(mom_256.index, mom_256, 0, where=mom_256 < 0, alpha=0.4, color='red')
ax2.set_title('2. 12-Month Cumulative Return → Signal Strength', fontsize=14, fontweight='bold')
ax2.set_ylabel('Cumulative Return')
ax2.grid(True, alpha=0.3)

# 3. Volatility-adjusted signal
ax3 = axes[1, 0]
vol = ewma_vol(asset_ret)
risk_adj = mom_256 / (vol * np.sqrt(256/252) + 1e-8)
ax3.plot(risk_adj.index, risk_adj, 'purple', lw=1.5)
ax3.axhline(0, color='black', lw=1)
ax3.axhline(2, color='green', ls='--', alpha=0.5)
ax3.axhline(-2, color='red', ls='--', alpha=0.5)
ax3.fill_between(risk_adj.index, risk_adj, 0, where=risk_adj > 0, alpha=0.4, color='green')
ax3.fill_between(risk_adj.index, risk_adj, 0, where=risk_adj < 0, alpha=0.4, color='red')
ax3.set_title('3. Risk-Adjusted Signal → Scaled by Volatility', fontsize=14, fontweight='bold')
ax3.set_ylabel('Risk-Adjusted Momentum')
ax3.grid(True, alpha=0.3)

# 4. Final forecast
ax4 = axes[1, 1]
fc = forecasts[asset].dropna()
ax4.plot(fc.index, fc, 'b-', lw=1.5)
ax4.axhline(0, color='black', lw=1)
ax4.axhline(10, color='green', ls='--', alpha=0.5, label='Strong Long')
ax4.axhline(-10, color='red', ls='--', alpha=0.5, label='Strong Short')
ax4.axhline(20, color='green', ls='-', alpha=0.3)
ax4.axhline(-20, color='red', ls='-', alpha=0.3)
ax4.fill_between(fc.index, fc.clip(lower=0), 0, alpha=0.4, color='green')
ax4.fill_between(fc.index, fc.clip(upper=0), 0, alpha=0.4, color='red')
ax4.set_title('4. Final Forecast → Scaled to [-20, +20]', fontsize=14, fontweight='bold')
ax4.set_ylabel('Forecast')
ax4.set_ylim(-25, 25)
ax4.legend(loc='upper right')
ax4.grid(True, alpha=0.3)

plt.suptitle('TSMOM Signal Generation Process (Gold Example)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('slide1_signal_generation.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
# 📈 SLIDE 2: Multi-Asset Diversification
---

In [ ]:
# SLIDE 2: Asset Class Diversification
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Asset class groupings
groups = {
    'Equities': ['QQQ', 'IWM', 'EFA', 'EEM', 'VGK', 'EWJ'],
    'Bonds': ['TLT', 'IEF', 'LQD', 'HYG'],
    'Commodities': ['GLD', 'SLV'],
    'Other': ['VNQ', 'UUP', 'FXE'],
}

group_colors = {'Equities': '#1f77b4', 'Bonds': '#2ca02c', 'Commodities': '#ff7f0e', 'Other': '#9467bd'}

# 1. Correlation heatmap
ax1 = axes[0, 0]
returns = prices.pct_change().dropna()
corr = returns.corr()
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, mask=mask, cmap='RdYlGn_r', center=0, annot=False,
            square=True, linewidths=0.5, ax=ax1, vmin=-0.5, vmax=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Correlation'})
ax1.set_title('Asset Correlation Matrix', fontsize=14, fontweight='bold')

# 2. Asset class pie chart
ax2 = axes[0, 1]
sizes = [len(v) for v in groups.values()]
colors_pie = [group_colors[k] for k in groups.keys()]
explode = (0.05, 0.05, 0.05, 0.05)
wedges, texts, autotexts = ax2.pie(sizes, explode=explode, labels=groups.keys(), 
                                    colors=colors_pie, autopct='%1.0f%%',
                                    shadow=True, startangle=90)
ax2.set_title('Universe Composition', fontsize=14, fontweight='bold')

# 3. Normalized prices by asset class
ax3 = axes[1, 0]
for group_name, assets in groups.items():
    available = [a for a in assets if a in prices.columns]
    if available:
        group_prices = prices[available].dropna()
        normalized = group_prices / group_prices.iloc[0]
        avg = normalized.mean(axis=1)
        ax3.plot(avg.index, avg, label=group_name, color=group_colors[group_name], lw=2)

ax3.set_title('Asset Class Performance (Normalized)', fontsize=14, fontweight='bold')
ax3.set_ylabel('Growth of $1')
ax3.legend(loc='upper left')
ax3.set_yscale('log')
ax3.grid(True, alpha=0.3)

# 4. Volatility by asset
ax4 = axes[1, 1]
vols = returns.std() * ANN * 100
colors_bar = []
for asset in vols.index:
    for group_name, assets in groups.items():
        if asset in assets:
            colors_bar.append(group_colors[group_name])
            break

bars = ax4.bar(range(len(vols)), vols.values, color=colors_bar)
ax4.set_xticks(range(len(vols)))
ax4.set_xticklabels(vols.index, rotation=45, ha='right')
ax4.set_ylabel('Annualized Volatility (%)')
ax4.set_title('Individual Asset Volatility', fontsize=14, fontweight='bold')
ax4.grid(True, alpha=0.3, axis='y')

# Legend
patches = [mpatches.Patch(color=c, label=g) for g, c in group_colors.items()]
ax4.legend(handles=patches, loc='upper right')

plt.suptitle('Multi-Asset Diversification: 15 Assets Across 4 Classes', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('slide2_diversification.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
# 📈 SLIDE 3: Strategy Performance Comparison
---

In [ ]:
# SLIDE 3: Main Performance Chart
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

strategies = {
    'TSMOM Trend': df['trend'],
    'S&P 500 (SPY)': df['spy'],
    '70/30 Blend': blend_70_30,
    '50/50 Blend': blend_50_50,
}

colors_strat = {
    'TSMOM Trend': COLORS['trend'],
    'S&P 500 (SPY)': COLORS['spy'],
    '70/30 Blend': COLORS['blend'],
    '50/50 Blend': COLORS['blend2'],
}

# 1. Cumulative returns
ax1 = axes[0, 0]
for name, ret in strategies.items():
    cum = (1 + ret).cumprod()
    ax1.semilogy(cum.index, cum, label=name, color=colors_strat[name], lw=2.5)

for crisis_name, (start, end) in CRISES.items():
    ax1.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.15, color='red')

ax1.axhline(1, color='gray', ls='--', alpha=0.5)
ax1.set_title('Cumulative Returns (Log Scale)', fontsize=14, fontweight='bold')
ax1.set_ylabel('Growth of $1')
ax1.legend(loc='upper left', fontsize=10)
ax1.grid(True, alpha=0.3)

# 2. Drawdowns
ax2 = axes[0, 1]
for name, ret in strategies.items():
    cum = (1 + ret).cumprod()
    roll_max = cum.expanding().max()
    dd = (cum / roll_max - 1) * 100
    ax2.plot(dd.index, dd, label=name, color=colors_strat[name], lw=1.5, alpha=0.8)

ax2.set_title('Drawdowns', fontsize=14, fontweight='bold')
ax2.set_ylabel('Drawdown (%)')
ax2.legend(loc='lower left', fontsize=10)
ax2.grid(True, alpha=0.3)

# 3. Rolling Sharpe
ax3 = axes[1, 0]
for name, ret in strategies.items():
    rs = ret.rolling(252).mean() / ret.rolling(252).std() * ANN
    ax3.plot(rs.index, rs, label=name, color=colors_strat[name], lw=1.5, alpha=0.8)

ax3.axhline(0, color='black', lw=1)
ax3.axhline(0.5, color='green', ls='--', alpha=0.5, label='Sharpe = 0.5')
ax3.set_title('Rolling 1-Year Sharpe Ratio', fontsize=14, fontweight='bold')
ax3.set_ylabel('Sharpe Ratio')
ax3.set_ylim(-2, 3)
ax3.legend(loc='upper right', fontsize=9)
ax3.grid(True, alpha=0.3)

# 4. Risk-Return scatter
ax4 = axes[1, 1]
for name, ret in strategies.items():
    r = ret.dropna()
    ann_ret = (1 + r).prod() ** (252 / len(r)) - 1
    ann_vol = r.std() * ANN
    ax4.scatter(ann_vol * 100, ann_ret * 100, s=300, c=colors_strat[name], 
                label=name, edgecolors='black', linewidths=2, zorder=5)
    ax4.annotate(name, (ann_vol * 100 + 0.5, ann_ret * 100 + 0.3), fontsize=10)

# Efficient frontier approximation
vols = np.linspace(5, 25, 100)
ax4.plot(vols, vols * 0.5, 'g--', alpha=0.5, label='Sharpe = 0.5')
ax4.plot(vols, vols * 0.7, 'b--', alpha=0.3, label='Sharpe = 0.7')

ax4.set_title('Risk-Return Profile', fontsize=14, fontweight='bold')
ax4.set_xlabel('Annualized Volatility (%)')
ax4.set_ylabel('Annualized Return (%)')
ax4.set_xlim(0, 25)
ax4.set_ylim(-5, 15)
ax4.legend(loc='upper left', fontsize=9)
ax4.grid(True, alpha=0.3)

plt.suptitle('Strategy Performance Comparison (2007-2024)', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('slide3_performance.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
# 📈 SLIDE 4: Crisis Alpha - The Key Value Proposition
---

In [ ]:
# SLIDE 4: Crisis Alpha
fig = plt.figure(figsize=(16, 10))
gs = GridSpec(2, 3, figure=fig, hspace=0.3, wspace=0.3)

# Calculate crisis returns
crisis_data = []
for crisis_name, (start, end) in CRISES.items():
    mask = (df.index >= pd.Timestamp(start)) & (df.index <= pd.Timestamp(end))
    if mask.sum() >= 5:
        trend_ret_crisis = (1 + df.loc[mask, 'trend']).prod() - 1
        spy_ret_crisis = (1 + df.loc[mask, 'spy']).prod() - 1
        crisis_data.append({
            'Crisis': crisis_name,
            'TSMOM': trend_ret_crisis,
            'SPY': spy_ret_crisis,
            'Excess': trend_ret_crisis - spy_ret_crisis,
        })

crisis_df = pd.DataFrame(crisis_data)

# 1. Main bar chart - Crisis returns comparison
ax1 = fig.add_subplot(gs[0, :])
x = np.arange(len(crisis_df))
width = 0.35

bars1 = ax1.bar(x - width/2, crisis_df['TSMOM'] * 100, width, label='TSMOM Trend', 
                color=COLORS['trend'], edgecolor='black', linewidth=1.5)
bars2 = ax1.bar(x + width/2, crisis_df['SPY'] * 100, width, label='S&P 500', 
                color=COLORS['spy'], edgecolor='black', linewidth=1.5)

ax1.axhline(0, color='black', lw=1)
ax1.set_xticks(x)
ax1.set_xticklabels(crisis_df['Crisis'], fontsize=11)
ax1.set_ylabel('Total Return (%)', fontsize=12)
ax1.set_title('🛡️ CRISIS ALPHA: TSMOM vs S&P 500 During Market Crashes', 
              fontsize=16, fontweight='bold')
ax1.legend(loc='upper right', fontsize=12)
ax1.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars1, crisis_df['TSMOM']):
    y = bar.get_height()
    ax1.annotate(f'{val*100:.1f}%', xy=(bar.get_x() + bar.get_width()/2, y),
                 xytext=(0, 5 if y >= 0 else -15), textcoords='offset points',
                 ha='center', fontsize=10, fontweight='bold')

for bar, val in zip(bars2, crisis_df['SPY']):
    y = bar.get_height()
    ax1.annotate(f'{val*100:.1f}%', xy=(bar.get_x() + bar.get_width()/2, y),
                 xytext=(0, -15 if y < 0 else 5), textcoords='offset points',
                 ha='center', fontsize=10, fontweight='bold')

# 2. Excess returns
ax2 = fig.add_subplot(gs[1, 0])
colors_excess = [COLORS['positive'] if x > 0 else COLORS['negative'] for x in crisis_df['Excess']]
ax2.barh(crisis_df['Crisis'], crisis_df['Excess'] * 100, color=colors_excess, 
         edgecolor='black', linewidth=1.5)
ax2.axvline(0, color='black', lw=1)
ax2.set_xlabel('Excess Return (%)')
ax2.set_title('Excess Return vs SPY', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='x')

for i, (crisis, excess) in enumerate(zip(crisis_df['Crisis'], crisis_df['Excess'])):
    ax2.annotate(f'+{excess*100:.1f}%', xy=(excess*100, i), 
                 xytext=(5 if excess > 0 else -35, 0), textcoords='offset points',
                 va='center', fontsize=10, fontweight='bold')

# 3. Scatter: Strategy vs Benchmark during crises
ax3 = fig.add_subplot(gs[1, 1])
ax3.scatter(crisis_df['SPY'] * 100, crisis_df['TSMOM'] * 100, 
            s=200, c=COLORS['trend'], edgecolors='black', linewidths=2, zorder=5)

for i, row in crisis_df.iterrows():
    ax3.annotate(row['Crisis'].split('\n')[0], 
                 (row['SPY']*100 + 1, row['TSMOM']*100 + 1), fontsize=9)

# Add diagonal line (where strategy = benchmark)
lims = [-60, 20]
ax3.plot(lims, lims, 'k--', alpha=0.5, label='Break-even')
ax3.fill_between(lims, lims, [lims[1], lims[1]], alpha=0.1, color='green', label='Outperform')

ax3.set_xlabel('S&P 500 Return (%)')
ax3.set_ylabel('TSMOM Return (%)')
ax3.set_title('Strategy vs Benchmark (Crisis Periods)', fontsize=12, fontweight='bold')
ax3.set_xlim(-60, 20)
ax3.set_ylim(-20, 20)
ax3.legend(loc='lower right')
ax3.grid(True, alpha=0.3)

# 4. Summary stats
ax4 = fig.add_subplot(gs[1, 2])
ax4.axis('off')

avg_excess = crisis_df['Excess'].mean() * 100
total_excess = crisis_df['Excess'].sum() * 100
win_rate = (crisis_df['Excess'] > 0).mean() * 100

text = f"""
📊 CRISIS ALPHA SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━

Average Excess Return:
+{avg_excess:.1f}%

Total Cumulative Excess:
+{total_excess:.0f}%

Win Rate (outperform):
{win_rate:.0f}%

Beta to SPY:
{np.cov(df['trend'], df['spy'])[0,1] / np.var(df['spy']):.2f}

━━━━━━━━━━━━━━━━━━━━━━━
✅ Trend provides INSURANCE
   when you need it most
"""

ax4.text(0.1, 0.5, text, transform=ax4.transAxes, fontsize=13,
         verticalalignment='center', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

plt.savefig('slide4_crisis_alpha.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
# 📈 SLIDE 5: Convexity Profile - Positive in Tails
---

In [ ]:
# SLIDE 5: Convexity Profile
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Return by decile
ax1 = axes[0]
deciles = pd.qcut(df['spy'], 10, labels=False, duplicates='drop')
avg_by_decile = df['trend'].groupby(deciles).mean() * DAYS

colors_decile = []
for i in avg_by_decile.index:
    if i <= 1:  # Worst deciles
        colors_decile.append('#2ca02c')  # Green (we want positive here!)
    elif i >= 8:  # Best deciles
        colors_decile.append('#2ca02c')  # Green
    else:
        colors_decile.append('#d62728' if avg_by_decile[i] < 0 else '#2ca02c')

bars = ax1.bar(avg_by_decile.index, avg_by_decile.values * 100, color=colors_decile,
               edgecolor='black', linewidth=1.5)
ax1.axhline(0, color='black', lw=1)
ax1.set_xlabel('S&P 500 Return Decile\n(0 = Worst 10%, 9 = Best 10%)', fontsize=11)
ax1.set_ylabel('Avg TSMOM Return (% Annualized)', fontsize=11)
ax1.set_title('🎯 CONVEXITY: Returns by Market Environment', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, axis='y')

# Add annotations
ax1.annotate('Crashes', xy=(0.5, -30), fontsize=12, fontweight='bold', color='red')
ax1.annotate('Rallies', xy=(8, -30), fontsize=12, fontweight='bold', color='green')

# 2. Scatter with regression
ax2 = axes[1]
x = df['spy'].values * 100
y = df['trend'].values * 100

# Color by magnitude
colors_scatter = np.where(np.abs(x) > np.percentile(np.abs(x), 90), COLORS['positive'], 'gray')
ax2.scatter(x, y, c=colors_scatter, alpha=0.5, s=10)

# Regression line
slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
x_line = np.linspace(x.min(), x.max(), 100)
ax2.plot(x_line, slope * x_line + intercept, 'r-', lw=2, 
         label=f'β = {slope:.2f}')

ax2.axhline(0, color='black', lw=0.5)
ax2.axvline(0, color='black', lw=0.5)
ax2.set_xlabel('S&P 500 Daily Return (%)', fontsize=11)
ax2.set_ylabel('TSMOM Daily Return (%)', fontsize=11)
ax2.set_title('Daily Return Relationship', fontsize=14, fontweight='bold')
ax2.legend(loc='upper left', fontsize=12)
ax2.grid(True, alpha=0.3)

# 3. Tail behavior
ax3 = axes[2]

# Calculate returns in different regimes
extreme_down = x < np.percentile(x, 5)  # Worst 5% of days
moderate_down = (x >= np.percentile(x, 5)) & (x < np.percentile(x, 25))
normal = (x >= np.percentile(x, 25)) & (x <= np.percentile(x, 75))
moderate_up = (x > np.percentile(x, 75)) & (x <= np.percentile(x, 95))
extreme_up = x > np.percentile(x, 95)  # Best 5% of days

regimes = ['Extreme\nDown\n(Worst 5%)', 'Moderate\nDown', 'Normal', 'Moderate\nUp', 'Extreme\nUp\n(Best 5%)']
masks = [extreme_down, moderate_down, normal, moderate_up, extreme_up]
avg_trend = [y[m].mean() * DAYS for m in masks]
avg_spy = [x[m].mean() * DAYS for m in masks]

x_pos = np.arange(len(regimes))
width = 0.35

bars1 = ax3.bar(x_pos - width/2, avg_trend, width, label='TSMOM', 
                color=COLORS['trend'], edgecolor='black')
bars2 = ax3.bar(x_pos + width/2, avg_spy, width, label='S&P 500', 
                color=COLORS['spy'], edgecolor='black')

ax3.axhline(0, color='black', lw=1)
ax3.set_xticks(x_pos)
ax3.set_xticklabels(regimes, fontsize=9)
ax3.set_ylabel('Avg Annualized Return (%)', fontsize=11)
ax3.set_title('Returns by Market Regime', fontsize=14, fontweight='bold')
ax3.legend(loc='upper left')
ax3.grid(True, alpha=0.3, axis='y')

plt.suptitle('TSMOM Convexity: Positive Returns in Extreme Environments', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('slide5_convexity.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
# 📈 SLIDE 6: Annual Returns Heatmap
---

In [ ]:
# SLIDE 6: Annual Returns Comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Annual returns
strats_annual = {
    'TSMOM': df['trend'],
    'SPY': df['spy'],
    '70/30': blend_70_30,
    '50/50': blend_50_50,
}

annual_df = pd.DataFrame()
for name, ret in strats_annual.items():
    annual = ret.resample('YE').apply(lambda x: (1 + x).prod() - 1)
    annual.index = annual.index.year
    annual_df[name] = annual

# 1. Heatmap
ax1 = axes[0]
sns.heatmap(annual_df.T * 100, annot=True, fmt='.0f', cmap='RdYlGn', center=0,
            linewidths=1, cbar_kws={'label': 'Annual Return (%)'}, ax=ax1,
            annot_kws={'fontsize': 9})
ax1.set_title('Annual Returns Heatmap (%)', fontsize=14, fontweight='bold')
ax1.set_xlabel('Year')
ax1.set_ylabel('Strategy')

# 2. Bar chart
ax2 = axes[1]
x = np.arange(len(annual_df))
width = 0.2

for i, (col, color) in enumerate(zip(annual_df.columns, 
                                      [COLORS['trend'], COLORS['spy'], COLORS['blend'], COLORS['blend2']])):
    ax2.bar(x + i*width, annual_df[col] * 100, width, label=col, color=color, 
            edgecolor='black', linewidth=0.5)

ax2.axhline(0, color='black', lw=1)
ax2.set_xticks(x + width * 1.5)
ax2.set_xticklabels(annual_df.index, rotation=45)
ax2.set_ylabel('Annual Return (%)')
ax2.set_title('Annual Returns by Strategy', fontsize=14, fontweight='bold')
ax2.legend(loc='upper left')
ax2.grid(True, alpha=0.3, axis='y')

plt.suptitle('Year-by-Year Performance Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('slide6_annual_returns.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
# 📈 SLIDE 7: Key Metrics Summary
---

In [ ]:
# SLIDE 7: Summary Metrics Table
fig, ax = plt.subplots(figsize=(14, 8))
ax.axis('off')

# Calculate all metrics
def calc_all_metrics(ret, name):
    r = ret.dropna()
    n = len(r)
    yrs = n / DAYS
    
    cum = (1 + r).cumprod()
    total = cum.iloc[-1] - 1
    cagr = (1 + total) ** (1/yrs) - 1
    vol = r.std() * ANN
    sharpe = r.mean() / r.std() * ANN if r.std() > 0 else 0
    
    down = r[r < 0]
    sortino = cagr / (down.std() * ANN) if len(down) > 0 else 0
    
    roll_max = cum.expanding().max()
    dd = cum / roll_max - 1
    max_dd = dd.min()
    calmar = cagr / abs(max_dd) if max_dd != 0 else 0
    
    # Correlation and beta to SPY
    if name != 'S&P 500':
        aligned = pd.DataFrame({'s': r, 'b': df['spy']}).dropna()
        corr = aligned['s'].corr(aligned['b'])
        beta = np.cov(aligned['s'], aligned['b'])[0,1] / np.var(aligned['b'])
    else:
        corr = 1.0
        beta = 1.0
    
    return {
        'Strategy': name,
        'CAGR': f'{cagr:.1%}',
        'Volatility': f'{vol:.1%}',
        'Sharpe': f'{sharpe:.2f}',
        'Sortino': f'{sortino:.2f}',
        'Max DD': f'{max_dd:.1%}',
        'Calmar': f'{calmar:.2f}',
        'Correlation': f'{corr:.2f}',
        'Beta': f'{beta:.2f}',
    }

metrics_list = [
    calc_all_metrics(df['trend'], 'TSMOM Trend'),
    calc_all_metrics(df['spy'], 'S&P 500'),
    calc_all_metrics(blend_70_30, '70/30 Blend'),
    calc_all_metrics(blend_50_50, '50/50 Blend'),
]

metrics_df = pd.DataFrame(metrics_list)

# Create table
table = ax.table(
    cellText=metrics_df.values,
    colLabels=metrics_df.columns,
    cellLoc='center',
    loc='center',
    colColours=['#4472C4'] * len(metrics_df.columns),
)

table.auto_set_font_size(False)
table.set_fontsize(12)
table.scale(1.2, 2.5)

# Style header
for i in range(len(metrics_df.columns)):
    table[(0, i)].set_text_props(color='white', fontweight='bold')

# Style cells
for i in range(1, len(metrics_df) + 1):
    for j in range(len(metrics_df.columns)):
        if i % 2 == 0:
            table[(i, j)].set_facecolor('#E6F2FF')
        else:
            table[(i, j)].set_facecolor('#FFFFFF')

ax.set_title('📊 Performance Metrics Summary (2007-2024)', fontsize=18, fontweight='bold', pad=20)

# Add footnote
footnote = """Notes: 
• Sharpe > 0.5 is considered good | Sortino accounts for downside risk only
• Calmar = CAGR / Max Drawdown | Beta < 1 indicates lower market sensitivity
• The 50/50 Blend has the HIGHEST Sharpe (0.72) and LOWEST Max DD (-26%)"""

ax.text(0.5, -0.1, footnote, transform=ax.transAxes, fontsize=10,
        ha='center', va='top', style='italic')

plt.savefig('slide7_metrics_table.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
# 📈 SLIDE 8: Why Use Futures? (Conceptual)
---

In [ ]:
# SLIDE 8: ETFs vs Futures Comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 1. Comparison table
ax1 = axes[0]
ax1.axis('off')

comparison_data = [
    ['Feature', 'ETFs (This Study)', 'Futures (Institutional)'],
    ['Transaction Costs', '~10 bps', '~2-5 bps'],
    ['Shorting', 'Borrow costs, restrictions', 'Native, no extra cost'],
    ['Leverage', 'Limited (margin req)', 'Efficient (5-15% margin)'],
    ['Roll Costs', 'Hidden in price (contango)', 'Explicit, manageable'],
    ['Expense Ratio', '0.03-0.75%', 'None'],
    ['Liquidity', 'Good', 'Excellent'],
    ['Asset Classes', 'Limited proxies', 'Direct exposure'],
    ['Expected Sharpe', '0.3-0.5', '0.4-0.7'],
]

table = ax1.table(
    cellText=comparison_data[1:],
    colLabels=comparison_data[0],
    cellLoc='center',
    loc='center',
    colColours=['#4472C4', '#70AD47', '#ED7D31'],
)

table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2.2)

for i in range(3):
    table[(0, i)].set_text_props(color='white', fontweight='bold')

ax1.set_title('ETFs vs Futures: Implementation Comparison', fontsize=14, fontweight='bold')

# 2. Expected improvement
ax2 = axes[1]

categories = ['Sharpe Ratio', 'CAGR\n(at 10% vol)', 'Transaction\nCosts', 'Roll Decay\nDrag']
etf_values = [0.44, 3.6, 0.4, 1.0]
futures_est = [0.55, 5.5, 0.1, 0.2]  # Estimated with futures

x = np.arange(len(categories))
width = 0.35

bars1 = ax2.bar(x - width/2, etf_values, width, label='ETF Implementation', 
                color=COLORS['spy'], edgecolor='black')
bars2 = ax2.bar(x + width/2, futures_est, width, label='Futures (Estimated)', 
                color=COLORS['blend'], edgecolor='black')

ax2.set_xticks(x)
ax2.set_xticklabels(categories)
ax2.set_ylabel('Value')
ax2.set_title('Expected Improvement with Futures', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, val in zip(bars1, etf_values):
    ax2.annotate(f'{val}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 xytext=(0, 3), textcoords='offset points', ha='center', fontsize=10)

for bar, val in zip(bars2, futures_est):
    ax2.annotate(f'{val}', xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                 xytext=(0, 3), textcoords='offset points', ha='center', fontsize=10)

plt.suptitle('📈 Why Professional CTAs Use Futures', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('slide8_futures_comparison.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
# 📈 SLIDE 9: Key Takeaways
---

In [ ]:
# SLIDE 9: Key Takeaways Visual
fig = plt.figure(figsize=(16, 10))

ax = fig.add_subplot(111)
ax.axis('off')

takeaways = """
╔══════════════════════════════════════════════════════════════════════════════════╗
║                          📚 KEY TAKEAWAYS                                         ║
╠══════════════════════════════════════════════════════════════════════════════════╣
║                                                                                   ║
║  1️⃣  TREND FOLLOWING IS CRISIS INSURANCE, NOT A RETURN ENGINE                    ║
║      • 0.44 Sharpe with ZERO beta to equities                                     ║
║      • Made money in 5/6 major crises (+25% to +56% excess)                       ║
║                                                                                   ║
║  2️⃣  THE BLEND IS THE WINNER                                                     ║
║      • 50/50 Blend: 0.72 Sharpe (best), -26% MaxDD (best)                         ║
║      • Gets most of equity upside + crisis protection                            ║
║                                                                                   ║
║  3️⃣  IMPLEMENTATION MATTERS                                                      ║
║      • Monthly rebalancing cut turnover from 51x to 4x                            ║
║      • Saved ~4% annual in transaction costs                                      ║
║      • Futures would improve results further (~0.5+ Sharpe expected)              ║
║                                                                                   ║
║  4️⃣  DIVERSIFICATION IS KEY                                                      ║
║      • 15 assets across 4 asset classes                                           ║
║      • Avoids concentration in any single market                                  ║
║                                                                                   ║
║  5️⃣  ACADEMIC EVIDENCE IS STRONG                                                 ║
║      • Moskowitz et al. (2012): TSMOM works across 58 markets                     ║
║      • Hurst et al. (2017): 100 years of positive returns                         ║
║                                                                                   ║
╚══════════════════════════════════════════════════════════════════════════════════╝
"""

ax.text(0.5, 0.5, takeaways, transform=ax.transAxes, fontsize=13,
        verticalalignment='center', horizontalalignment='center',
        fontfamily='monospace',
        bbox=dict(boxstyle='round', facecolor='white', edgecolor='#4472C4', linewidth=3))

plt.savefig('slide9_takeaways.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.show()

---
# 📁 Save All Images
---

In [ ]:
import os

# List all saved images
images = [f for f in os.listdir('.') if f.endswith('.png')]

print("📊 Generated Presentation Images:")
print("=" * 50)
for img in sorted(images):
    print(f"  ✅ {img}")

print("\n" + "=" * 50)
print("Copy these images to your PowerPoint presentation!")

---

## 📚 References

1. **Moskowitz, Ooi, Pedersen (2012)** - "Time Series Momentum" - *Journal of Financial Economics*
2. **Hurst, Ooi, Pedersen (2017)** - "A Century of Evidence on Trend-Following Investing" - *Journal of Portfolio Management*
3. **Baltas & Kosowski (2020)** - "Demystifying Time-Series Momentum Strategies" - *Journal of Financial Markets*
4. **Carver (2015)** - "Systematic Trading" - *Book*

---

*Commodities Club - Northeastern University - Session 4*